# Setup

In [ ]:
from pprint import pprint
import os, math, time
import pandas as pd
import torch
import torch.optim as optim
from transformers.utils import logging
from transformers import set_seed

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = device.type == 'cuda' and torch.cuda.is_bf16_supported()

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# Seed.
seed = 42
set_seed(seed)

# 1. Scaling Laws

- **Core idea:** [more resources → generally lower loss](https://arxiv.org/abs/2001.08361), following approximate power laws
  - **Model size (`N`):** parameter count
  - **Data size (`D`):** training tokens
  - **[Compute (`C`)](https://arxiv.org/abs/2203.15556):** roughly `6 × N × D` FLOPs for dense Transformer training
  - **Fixed budget:** larger model → fewer affordable training tokens
  - **Takeaway:** balance model size and data; bigger ≠ automatically better

In [ ]:
# Budget trade-off only—not a prediction of model quality.
compute_budget = 6e15  # Illustrative FLOPs budget

for parameters in [10_000_000, 20_000_000, 40_000_000]:
    training_tokens = compute_budget / (6 * parameters)

    print(
        f"Parameters: {parameters / 1e6:.0f}M | "
        f"Training tokens: {training_tokens / 1e6:.0f}M"
    )
# 10M → 100M tokens; 20M → 50M tokens; 40M → 25M tokens

# 2. Mixture of Experts — MoE

- **Core idea:** [multiple expert FFNs; only a few activated per token](https://arxiv.org/abs/2401.04088)
  - **Router:** select experts; assign combination weights
  - **Experts:** sub-networks—not complete LLMs
  - **Output:** weighted combination of selected expert outputs
  - **Benefit:** more total parameters without computing through all of them
  - **Cost:** all weights still need storage; routing and communication overhead

In [ ]:
import torch
import torch.nn as nn

# One token; four untrained experts; select two.
x = torch.randn(4)

router = nn.Linear(4, 4)
experts = nn.ModuleList([
    nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 4))
    for _ in range(4)
])

scores = router(x)
top_scores, expert_ids = scores.topk(2)
weights = top_scores.softmax(dim=0)

output = torch.zeros_like(x)
for weight, expert_id in zip(weights, expert_ids.tolist()):
    expert_output = experts[expert_id](x)
    output = output + weight * expert_output

print("Selected experts:", expert_ids.tolist())
print("Output:", output.detach())

# 3. Long-Context LLMs

- **Core idea:** longer inputs, effective information use
  - **[RoPE scaling](https://arxiv.org/abs/2309.00071):** extend position handling; often additional training
  - **Full attention:** `O(L²)` computation; `L` = sequence length
  - **[FlashAttention](https://arxiv.org/abs/2205.14135):** reduced memory traffic/storage; still quadratic computation
  - **[Limitation](https://arxiv.org/abs/2307.03172):** larger context window ≠ reliable recall

In [ ]:
# Naive attention-score matrix only: one sequence, one head, FP16.
# Not total model memory; FlashAttention avoids storing this full matrix.
bytes_per_value = 2

for length in [1024, 2048, 4096]:
    score_count = length * length
    memory_mib = score_count * bytes_per_value / 1024**2

    print(f"{length} tokens: {memory_mib:.0f} MiB")

# 1024 → 2 MiB; 2048 → 8 MiB; 4096 → 32 MiB

# 4. Reasoning / Test-Time Compute / Verifiers

- **Core idea:** [more computation while answering](https://arxiv.org/abs/2408.03314); usually fixed model weights
  - **Sequential:** intermediate steps → checking → revision
  - **Parallel:** multiple candidate solutions → selection
  - **Outcome verifier:** evaluate final answers
  - **Process verifier:** evaluate intermediate steps
  - **Trade-off:** extra compute/latency; correctness still not guaranteed

In [ ]:
# Solve x + 7 = 12.
# Mock model-generated candidates; exact rule-based verification.
candidates = [4, 6, 5]
verified_answers = []

for answer in candidates:
    is_correct = answer + 7 == 12
    if is_correct:
        verified_answers.append(answer)

selected = verified_answers[0] if verified_answers else None

print("Candidates:", candidates)
print("Verified answer:", selected)

# 5. Distillation + Synthetic Data

- **Core idea:** teacher behavior transfer + generated training examples
  - **[Distillation](https://arxiv.org/abs/1503.02531):** teacher → student, often smaller
    - **Soft targets:** match teacher probabilities
    - **Response targets:** imitate teacher-generated answers
  - **[Synthetic data](https://arxiv.org/abs/2212.10560):** model-generated tasks/answers
    - **Pipeline:** generate → filter/deduplicate → train
  - **Distinction:** distillation = behavior transfer; synthetic data = data origin
  - **Risk:** inherited teacher mistakes; quality control essential

In [ ]:
import torch
import torch.nn.functional as F

# Toy next-token distillation: three possible tokens.
# Teacher probabilities supplied manually; student = trainable logits.
teacher_probs = torch.tensor([[0.7, 0.2, 0.1]])
student_logits = torch.nn.Parameter(torch.zeros(1, 3))
optimizer = torch.optim.SGD([student_logits], lr=0.5)

student_probs = student_logits.detach().softmax(dim=-1)
print("Before:", student_probs)

for _ in range(100):
    student_log_probs = student_logits.log_softmax(dim=-1)
    loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

student_probs = student_logits.detach().softmax(dim=-1)
print("Teacher:", teacher_probs)
print("After:", student_probs)

# 6. Modern Post-Training

- **Core idea:** adapt pretrained behavior through demonstrations, preferences, rewards
  - **[SFT](https://arxiv.org/abs/2203.02155):** imitate demonstrated responses
  - **[DPO](https://arxiv.org/abs/2305.18290):** preferred/rejected pairs; no separate reward model or RL rollout loop
  - **[RLHF](https://arxiv.org/abs/2203.02155):** optimize rewards learned from human feedback
  - **[RLVR](https://arxiv.org/abs/2501.12948):** checkable rewards—math answers, passing tests
  - **[PPO](https://arxiv.org/abs/1707.06347) / [GRPO](https://arxiv.org/abs/2402.03300):** RL optimization algorithms—not feedback sources
    - **GRPO:** group-relative rewards; no separate learned value model
  - **Possible path:** pretraining → SFT → DPO or RL; not mandatory